In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
from scipy.stats import norm
from datetime import datetime
ticker="AAPL"
stock=yf.Ticker(ticker)
current_price=stock.history(period="1d")["Close"].iloc[-1]
hist=stock.history(period="1y")
hist["Log_Return"]=np.log(hist["Close"]/hist["Close"].shift(1))
sigma=hist["Log_Return"].std()*np.sqrt(252)
tnx=yf.Ticker("^TNX")
risk_free=tnx.history(period="1d")["Close"].iloc[-1]/100
expiry=stock.options[-1]
chain=stock.option_chain(expiry)
calls=chain.calls
puts=chain.puts
atm_idx=(calls["strike"]-current_price).abs().idxmin()
K=calls.loc[atm_idx,"strike"]
expiry_date=datetime.strptime(expiry,"%Y-%m-%d")
T=(expiry_date-datetime.now()).total_seconds()/(365*24*60*60)
r=risk_free
S=current_price
d1=(np.log(S/K)+(r+sigma**2/2)*T)/(sigma*np.sqrt(T))
d2=d1-sigma*np.sqrt(T)
call_delta=norm.cdf(d1)
put_delta=norm.cdf(d1)-1
gamma=norm.pdf(d1)/(S*sigma*np.sqrt(T))
call_theta=(-(S*norm.pdf(d1)*sigma)/(2*np.sqrt(T))-r*K*np.exp(-r*T)*norm.cdf(d2))
put_theta=(-(S*norm.pdf(d1)*sigma)/(2*np.sqrt(T))+r*K*np.exp(-r*T)*norm.cdf(-d2))
vega=S*np.sqrt(T)*norm.pdf(d1)
greeks=pd.DataFrame({
    "Greek":["Delta","Gamma","Theta","Vega"],
    "ATM Call":[call_delta,gamma,call_theta,vega],
    "ATM Put":[put_delta,gamma,put_theta,vega]
})
print("Underlying Price:",round(S,2))
print("ATM Strike:",K)
print("Volatility:",round(sigma,4))
print("Risk-Free Rate:",round(r,4))
print("Expiry:",expiry)
print("Time to Maturity:",round(T,4))
print("\nGreeks Table\n")
print(greeks.round(4))